# Generación de Archivos de Datos para Visualización Auto
Notebook para la preparación de los archivos JSON para la visualización de datos de resultados de la encuesta m195 Encuesta Bienal de Culturas - Encuesta de Prácticas Artísticas, Culturales, Creativas y Patrimoniales 2025.

- Por: javier.ojeda@scrd.gov.co
- Fecha: 2026-05-13

In [88]:
import pandas as pd

In [89]:
# Parametrización y configuración

codigo_medicion = 'm195'
columna_edad = 'D3'
columna_localidad = 'V1'
columna_grupo_edad = 'D3_1'
columna_sexo = 'D1'
columna_clase = 'Clase'
columna_factor_expansion = 'FACTOR'
folder_output = f'D:\\development\\javascript\\observatorio\\public\\content\\mediciones\\{codigo_medicion}'

# Si se han hecho cambios en las variables seleccionadas
actualizar_opciones = False
actualizar_respuestas = False

In [90]:
# DF del dataset principal de respuestas de la encuesta
df_respuestas_detalle = pd.read_excel(f'{codigo_medicion}_datos.xlsx',sheet_name='datos')

# Seleccionar respuestas solo de 13 años o más
df_respuestas_detalle = df_respuestas_detalle[df_respuestas_detalle[columna_edad] >= 13]

In [301]:
# Otros elementos de la encuesta
df_secciones = pd.read_excel(f'{codigo_medicion}_datos.xlsx',sheet_name='secciones')
df_variables = pd.read_excel(f'{codigo_medicion}_datos.xlsx',sheet_name='variables')
df_preguntas = pd.read_excel(f'{codigo_medicion}_datos.xlsx',sheet_name='preguntas')
df_ficha_tecnica = pd.read_excel(f'{codigo_medicion}_datos.xlsx',sheet_name='ficha_tecnica')
df_creditos = pd.read_excel(f'{codigo_medicion}_datos.xlsx',sheet_name='creditos')

In [302]:
# Cargar df_localidades
df_localidades = pd.read_excel(f'{codigo_medicion}_datos.xlsx',sheet_name='localidades')
df_localidades = df_localidades[['localidad_cod', 'localidad_residencia', 'latitud', 'longitud', 'es_localidad', 'poblacion']]

# Filtrar solo localidades, localidad.es_localidad == 1
df_localidades = df_localidades[df_localidades['es_localidad'] == 1]

In [303]:
# Identificar variables que se incluyen en el cubo, es decir, aquellas que tengan en_cubo == 1
df_variables = df_variables[df_variables['en_cubo'] == 1]

columnas_a_derretir = df_variables['codigo_variable'].tolist()

# Crear un nuevo df con solo las columnas a derretir
df_respuestas_a_melt = df_respuestas_detalle[columnas_a_derretir]

In [304]:
# Derretir hoja de respuestas
df_respuestas = (
    df_respuestas_a_melt
        .melt(
            id_vars=[columna_factor_expansion],
            var_name='codigo_variable',
            value_name='respuesta'
        )
        .groupby(['codigo_variable', 'respuesta'])
        .agg(
            suma_factor=('FACTOR', 'sum'),
            cantidad_respuestas=('FACTOR', 'count')
        )
        .reset_index()
)

In [305]:
# Agregar datos de variables, indice_pregunta e indice_variable
df_respuestas = df_respuestas.merge(df_variables[['codigo_variable', 'indice_pregunta', 'indice_variable']], on='codigo_variable', how='left')

# Ordenar por suma_factor de mayor a menor
df_respuestas = df_respuestas.sort_values(by='suma_factor', ascending=False)

# Ordenar por indice_pregunta e indice_variable
df_respuestas = df_respuestas.sort_values(by=['indice_pregunta', 'indice_variable'], ascending=[True, True])

# Reordenar columnas
df_respuestas = df_respuestas[['codigo_variable', 'indice_pregunta', 'indice_variable', 'respuesta', 'suma_factor', 'cantidad_respuestas']]

In [306]:
# Leer archivo de respuestas y formato del archivo de opciones
df_opciones = pd.read_excel(f'{codigo_medicion}_opciones.xlsx',sheet_name='respuestas')

# Agregar a df_respuestas la columna respuesta_v2 y respuesta_number buscando coinciencia en indice_pregunta y respuesta
df_respuestas = df_respuestas.merge(df_opciones[['indice_pregunta', 'respuesta', 'respuesta_v2', 'respuesta_number']], on=['indice_pregunta', 'respuesta'], how='left')
# Reordenar columnas
df_respuestas = df_respuestas[['codigo_variable', 'indice_pregunta', 'indice_variable', 'respuesta', 'respuesta_v2', 'respuesta_number', 'suma_factor', 'cantidad_respuestas']]

## Creación de respuestas por localidad


In [307]:
# Derretir respuestas agregando la dimensión de localidad
df_respuestas_localidad = (
    df_respuestas_a_melt
        .melt(
            id_vars=[columna_factor_expansion, columna_localidad],
            var_name='codigo_variable',
            value_name='respuesta'
        )
        .groupby([columna_localidad, 'codigo_variable', 'respuesta'])
        .agg(
            suma_factor=(columna_factor_expansion, 'sum'),
            cantidad_respuestas=(columna_factor_expansion, 'count')
        )
        .reset_index()
)

In [308]:
# Agregar datos de variables, indice_pregunta e indice_variable
df_respuestas_localidad = df_respuestas_localidad.merge(df_variables[['codigo_variable', 'indice_pregunta', 'indice_variable']], on='codigo_variable', how='left')

# Ordenar por suma_factor de mayor a menor
df_respuestas_localidad = df_respuestas_localidad.sort_values(by='suma_factor', ascending=False)

# Ordenar por indice_pregunta e indice_variable
df_respuestas_localidad = df_respuestas_localidad.sort_values(by=['indice_pregunta', 'indice_variable'], ascending=[True, True])

# Cambiar el nombre de la columna localidad por localidad_v2
df_respuestas_localidad = df_respuestas_localidad.rename(columns={columna_localidad: 'localidad'}) 

# Cambiar el nombre de "Código localidad" a "localidad_cod"
df_respuestas_localidad = df_respuestas_localidad.rename(columns={'localidad': 'localidad_cod'})

# Establecer localidad_cod como un numero entero, evita errores
df_respuestas_localidad['localidad_cod'] = df_respuestas_localidad['localidad_cod'].fillna(0).astype(int)

# Reordenar columnas
# df_respuestas_localidad = df_respuestas_localidad[['localidad_cod', 'codigo_variable', 'indice_pregunta', 'indice_variable', 'respuesta', 'suma_factor', 'cantidad_respuestas']]

In [309]:
# Agregar a df_respuestas la columna respuesta_v2 y respuesta_number buscando coinciencia en indice_pregunta y respuesta
df_respuestas_localidad = df_respuestas_localidad.merge(df_opciones[['indice_pregunta', 'respuesta', 'respuesta_v2', 'respuesta_number']], on=['indice_pregunta', 'respuesta'], how='left')

# Reordenar columnas
df_respuestas_localidad = df_respuestas_localidad[
    [
        'localidad_cod', 'codigo_variable', 'indice_pregunta',
        'indice_variable', 'respuesta', 'respuesta_v2', 'respuesta_number',
        'suma_factor', 'cantidad_respuestas']
]

## Creación de respuestas por Grupo de Edad

In [310]:
# Cargar df_edades
df_edades = pd.read_excel(f'{codigo_medicion}_datos.xlsx',sheet_name='edades')
df_edades = df_edades[['edad', 'grupo_edad_pp', 'grupo_edad_pp_cod']]

In [311]:
# Derretir respuestas agregando la dimensión de localidad
df_respuestas_edad = (
    df_respuestas_a_melt
        .melt(
            id_vars=[columna_factor_expansion, columna_grupo_edad],
            var_name='codigo_variable',
            value_name='respuesta'
        )
        .groupby([columna_grupo_edad, 'codigo_variable', 'respuesta'])
        .agg(
            suma_factor=(columna_factor_expansion, 'sum'),
            cantidad_respuestas=(columna_factor_expansion, 'count')
        )
        .reset_index()
)

In [312]:
# Agregar datos de variables, indice_pregunta e indice_variable
df_respuestas_edad = df_respuestas_edad.merge(df_variables[['codigo_variable', 'indice_pregunta', 'indice_variable']], on='codigo_variable', how='left')

# Ordenar por suma_factor de mayor a menor
df_respuestas_edad = df_respuestas_edad.sort_values(by='suma_factor', ascending=False)

# Ordenar por indice_pregunta e indice_variable
df_respuestas_edad = df_respuestas_edad.sort_values(by=['indice_pregunta', 'indice_variable'], ascending=[True, True])

# Cambiar el nombre de la columna grupo_edad por grupo_edad_pp_cod
df_respuestas_edad = df_respuestas_edad.rename(columns={columna_grupo_edad: 'grupo_edad_pp_cod'}) 

# Establecer grupo_edad_pp_cod como un numero entero, evita errores
df_respuestas_edad['grupo_edad_pp_cod'] = df_respuestas_edad['grupo_edad_pp_cod'].fillna(0).astype(int)

In [313]:
# Agregar a df_respuestas la columna respuesta_v2 y respuesta_number buscando coinciencia en indice_pregunta y respuesta
df_respuestas_edad = df_respuestas_edad.merge(df_opciones[['indice_pregunta', 'respuesta', 'respuesta_v2', 'respuesta_number']], on=['indice_pregunta', 'respuesta'], how='left')

# Reordenar columnas
df_respuestas_edad = df_respuestas_edad[
    [
        'grupo_edad_pp_cod', 'codigo_variable', 'indice_pregunta',
        'indice_variable', 'respuesta', 'respuesta_v2', 'respuesta_number',
        'suma_factor', 'cantidad_respuestas']
]

## Creación de respuestas por Sexo

In [314]:
# Derretir respuestas agregando la dimensión de sexo
df_respuestas_sexo= (
    df_respuestas_a_melt
        .melt(
            id_vars=[columna_factor_expansion, columna_sexo],
            var_name='codigo_variable',
            value_name='respuesta'
        )
        .groupby([columna_sexo, 'codigo_variable', 'respuesta'])
        .agg(
            suma_factor=(columna_factor_expansion, 'sum'),
            cantidad_respuestas=(columna_factor_expansion, 'count')
        )
        .reset_index()
)

In [315]:
# Agregar datos de variables, indice_pregunta e indice_variable
df_respuestas_sexo = df_respuestas_sexo.merge(df_variables[['codigo_variable', 'indice_pregunta', 'indice_variable']], on='codigo_variable', how='left')

# Ordenar por suma_factor de mayor a menor
df_respuestas_sexo = df_respuestas_sexo.sort_values(by='suma_factor', ascending=False)

# Ordenar por indice_pregunta e indice_variable
df_respuestas_sexo = df_respuestas_sexo.sort_values(by=['indice_pregunta', 'indice_variable'], ascending=[True, True])

# Cambiar el nombre de la columna grupo_edad por grupo_edad_pp_cod
df_respuestas_sexo = df_respuestas_sexo.rename(columns={columna_sexo: 'sexo'}) 

# Establecer sexo como un numero entero, evita errores
df_respuestas_sexo['sexo'] = df_respuestas_sexo['sexo'].fillna(0).astype(int)

# Agregar a df_respuestas la columna respuesta_v2 y respuesta_number buscando coinciencia en indice_pregunta y respuesta
df_respuestas_sexo = df_respuestas_sexo.merge(df_opciones[['indice_pregunta', 'respuesta', 'respuesta_v2', 'respuesta_number']], on=['indice_pregunta', 'respuesta'], how='left')

# Reordenar columnas
df_respuestas_sexo = df_respuestas_sexo[
    [
        'sexo', 'codigo_variable', 'indice_pregunta',
        'indice_variable', 'respuesta', 'respuesta_v2', 'respuesta_number',
        'suma_factor', 'cantidad_respuestas']
]

## Creación de respuestas por Clase (Urbano/Rural)

In [316]:
# Derretir respuestas agregando la dimensión de clase (urbano y rural)
df_respuestas_clase= (
    df_respuestas_a_melt
        .melt(
            id_vars=[columna_factor_expansion, columna_clase],
            var_name='codigo_variable',
            value_name='respuesta'
        )
        .groupby([columna_clase, 'codigo_variable', 'respuesta'])
        .agg(
            suma_factor=(columna_factor_expansion, 'sum'),
            cantidad_respuestas=(columna_factor_expansion, 'count')
        )
        .reset_index()
)

In [317]:
# Agregar datos de variables, indice_pregunta e indice_variable
df_respuestas_clase = df_respuestas_clase.merge(df_variables[['codigo_variable', 'indice_pregunta', 'indice_variable']], on='codigo_variable', how='left')

# Ordenar por suma_factor de mayor a menor
df_respuestas_clase = df_respuestas_clase.sort_values(by='suma_factor', ascending=False)

# Ordenar por indice_pregunta e indice_variable
df_respuestas_clase = df_respuestas_clase.sort_values(by=['indice_pregunta', 'indice_variable'], ascending=[True, True])

# Cambiar el nombre de la columna grupo_edad por grupo_edad_pp_cod
df_respuestas_clase = df_respuestas_clase.rename(columns={columna_clase: 'clase'}) 

# Establecer clase como un numero entero, evita errores
df_respuestas_clase['clase'] = df_respuestas_clase['clase'].fillna(0).astype(int)

# Agregar a df_respuestas la columna respuesta_v2 y respuesta_number buscando coinciencia en indice_pregunta y respuesta
df_respuestas_clase = df_respuestas_clase.merge(df_opciones[['indice_pregunta', 'respuesta', 'respuesta_v2', 'respuesta_number']], on=['indice_pregunta', 'respuesta'], how='left')

# Reordenar columnas
df_respuestas_clase = df_respuestas_clase[
    [
        'clase', 'codigo_variable', 'indice_pregunta',
        'indice_variable', 'respuesta', 'respuesta_v2', 'respuesta_number',
        'suma_factor', 'cantidad_respuestas']
]

## Exportar datos en formato JSON

In [318]:
df_secciones.to_json(f'{folder_output}\\secciones.json', orient='records', force_ascii=False, indent=4)
df_preguntas.to_json(f'{folder_output}\\preguntas.json', orient='records', force_ascii=False, indent=4)
df_variables.to_json(f'{folder_output}\\variables.json', orient='records', force_ascii=False, indent=4)
df_respuestas.to_json(f'{folder_output}\\respuestas.json', orient='records', force_ascii=False, indent=4)
df_ficha_tecnica.to_json(f'{folder_output}\\ficha_tecnica.json', orient='records', force_ascii=False, indent=4)
df_creditos.to_json(f'{folder_output}\\creditos.json', orient='records', force_ascii=False, indent=4)
df_localidades.to_json(f'{folder_output}\\localidades.json', orient='records', force_ascii=False, indent=4)
df_respuestas_localidad.to_json(f'{folder_output}\\respuestas_localidad.json', orient='records', force_ascii=False, indent=4)
df_respuestas_edad.to_json(f'{folder_output}\\respuestas_edad.json', orient='records', force_ascii=False, indent=4)
df_respuestas_sexo.to_json(f'{folder_output}\\respuestas_sexo.json', orient='records', force_ascii=False, indent=4)
df_respuestas_clase.to_json(f'{folder_output}\\respuestas_clase.json', orient='records', force_ascii=False, indent=4)

In [319]:
if actualizar_respuestas:

    # Sobre escribir archivo
    from openpyxl import load_workbook

    # Ruta del archivo Excel
    respuestas_file_path = f'{codigo_medicion}_respuestas.xlsx'

    # Utilizamos el engine openpyxl para abrir el archivo sin sobrescribir las otras hojas
    with pd.ExcelWriter(respuestas_file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        # Escribimos o actualizamos la hoja de cálculo 'respuestas_detalle'
        df_respuestas_localidad.to_excel(writer, index=False, sheet_name='respuestas_localidad')
        df_respuestas_edad.to_excel(writer, index=False, sheet_name='respuestas_edad')
        df_respuestas_sexo.to_excel(writer, index=False, sheet_name='respuestas_sexo')
        df_respuestas_clase.to_excel(writer, index=False, sheet_name='respuestas_clase')

## Generación de datos para control de formato de respuestas

In [320]:
df_respuestas_agrupadas = pd.DataFrame()

# Crear DataFrame agrupando solo las columnas indice_pregunta y respuesta
df_respuestas_agrupadas = df_respuestas[['indice_pregunta', 'respuesta']]

# Agregar columna de preguntas['etiqueta_1'] a df_respuestas_agrupadas, relacionando por indice_pregunta
df_respuestas_agrupadas = df_respuestas_agrupadas.merge(df_preguntas[['indice_pregunta', 'etiqueta_1']], on='indice_pregunta', how='left')

# Ordenar por indice_pregunta y luego por respuesta
df_respuestas_agrupadas = df_respuestas_agrupadas.sort_values(by=['indice_pregunta', 'respuesta'], ascending=[True, True])

# Ordenar columnas así: indice_pregunta, etiqueta_1, respuesta
df_respuestas_agrupadas = df_respuestas_agrupadas[['indice_pregunta', 'etiqueta_1', 'respuesta']]

# Eliminar filas repetidas
df_respuestas_agrupadas = df_respuestas_agrupadas.drop_duplicates()

df_respuestas_agrupadas.shape

(1327, 3)

In [321]:
if actualizar_opciones:
    # Sobre escribir archivo
    from openpyxl import load_workbook

    # Ruta del archivo Excel
    file_path = f'{codigo_medicion}_opciones.xlsx'

    # Cargar el archivo Excel existente
    book = load_workbook(file_path)

    # Utilizamos el engine openpyxl para abrir el archivo sin sobrescribir las otras hojas
    with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        # Escribimos o actualizamos la hoja de cálculo 'respuestas_detalle'
        df_respuestas_agrupadas.to_excel(writer, index=False, sheet_name='respuestas_org')